# seisfetch: Quickstart

This notebook walks through the core API of `seisfetch`:

1. Fetch waveforms from EarthScope S3 → numpy arrays
2. Auto-routing: CI → SCEDC, BK → NCEDC, IU → EarthScope
3. FDSN web services for non-S3 providers (GEOFON, INGV, ...)
4. Output formats: numpy, xarray, ObsPy Stream
5. Parse local miniSEED files

**Core deps: numpy + boto3 + pymseed.  No ObsPy needed for steps 1–3.**

## Install

```bash
pip install -e "."                        # core only
pip install -e ".[xarray,obspy]"          # with xarray + ObsPy interop
```

In [2]:
import numpy as np

from seisfetch import (
    SeisfetchClient,
    bundle_to_xarray,
    list_providers,
    parse_mseed,
    route_network,
)

ModuleNotFoundError: No module named 'seisfetch'

## 1. Fetch from EarthScope S3 → numpy

The default backend is `s3_open` — anonymous access to three S3 buckets.
Times can be ISO strings, epoch floats, or `datetime` objects (no ObsPy needed).

In [1]:
client = SeisfetchClient(backend="s3_open")

# IU.ANMO → auto-routes to earthscope-geophysical-data
bundle = client.get_numpy(
    "IU",
    "ANMO",
    starttime="2024-01-15T00:00:00",
    endtime="2024-01-15T00:10:00",
)

print(f"Traces: {len(bundle)}")
print(f"Channel IDs: {bundle.ids}")

# Get as a plain dict of numpy arrays
arrays = bundle.to_dict()
for nslc, data in arrays.items():
    print(f"  {nslc}: shape={data.shape}, dtype={data.dtype}")

NameError: name 'SeisfetchClient' is not defined

## 2. Auto-routing by network code

The client automatically picks the fastest S3 bucket for each network:

| Network | Datacenter | S3 Bucket |
|---------|-----------|-----------|
| CI | SCEDC | scedc-pds (us-west-2) |
| BK | NCEDC | ncedc-pds (us-east-2) |
| IU, UW, TA, ... | EarthScope | earthscope-geophysical-data (us-east-2) |

In [ ]:
# Check routing for any network
for net in ["CI", "BK", "NC", "IU", "UW", "TA"]:
    print(f"  {net} → {route_network(net)}")

In [ ]:
# SCEDC and NCEDC are per-channel archives — specify channel explicitly
bundle_ci = client.get_numpy(
    "CI",
    "SDD",
    channel="BHZ",
    starttime="2024-06-01T00:00:00",
    endtime="2024-06-01T00:10:00",
)
print(f"CI.SDD.BHZ: {bundle_ci.to_dict().get('CI.SDD..BHZ', np.array([])).shape}")

## 3. FDSN web services

For providers without S3 buckets (GEOFON, INGV, ETH, ...),
use `backend="fdsn"`.  Downloads use HTTP directly — never ObsPy.

In [ ]:
# List all 37+ providers
providers = list_providers()
print(f"{len(providers)} FDSN providers available:")
for name in sorted(providers)[:10]:
    print(f"  {name:20s} {providers[name]}")
print("  ...")

In [ ]:
# Fetch from GEOFON (GFZ Potsdam)
fdsn_client = SeisfetchClient(backend="fdsn", providers="GEOFON")
bundle_ge = fdsn_client.get_numpy(
    "GE",
    "DAV",
    channel="BHZ",
    location="*",
    starttime="2024-06-01T00:00:00",
    endtime="2024-06-01T00:10:00",
)
print(f"GE.DAV via GEOFON: {len(bundle_ge)} traces")

## 4. Output: xarray Dataset

Convert any TraceBundle to an xarray Dataset with `datetime64[ns]` coordinates.
Requires: `pip install xarray`

In [ ]:
ds = bundle_to_xarray(bundle)
print(ds)

# Each channel is a DataArray with time coordinate
for var in ds.data_vars:
    da = ds[var]
    print(f"\n{var}:")
    print(f"  shape: {da.shape}")
    print(f"  time range: {da.time.values[0]} → {da.time.values[-1]}")
    print(f"  sampling_rate: {da.attrs['sampling_rate']} Hz")

## 5. Output: xarray via client shortcut

In [ ]:
ds_direct = client.get_xarray(
    "IU",
    "ANMO",
    channel="BHZ",
    location="00",
    starttime="2024-01-15T00:00:00",
    endtime="2024-01-15T00:10:00",
)
print(ds_direct)

## 6. Output: ObsPy Stream (optional interop)

When you need ObsPy processing (filtering, response removal, plotting),
convert with `get_waveforms()` or `bundle_to_obspy()`.

**Requires: `pip install obspy`**

Note: downloading and decoding still uses S3 + pymseed — ObsPy only
wraps the final numpy arrays into `Trace` objects.

In [ ]:
try:
    from seisfetch import bundle_to_obspy

    st = bundle_to_obspy(bundle)
    print(st)
    # st.filter("bandpass", freqmin=0.1, freqmax=2.0)
    # st.plot()
except ImportError:
    print("ObsPy not installed — skipping.  pip install obspy")

## 7. Parse local miniSEED files

`parse_mseed()` works on any raw bytes — from S3, HTTP, or local disk.

In [ ]:
# Simulate reading a local file
raw_bytes = client.get_raw(
    "IU",
    "ANMO",
    starttime="2024-01-15",
    endtime="2024-01-15T00:01:00",
)
print(f"Raw miniSEED: {len(raw_bytes):,} bytes")

# Parse → numpy
bundle_local = parse_mseed(raw_bytes)
print(f"Parsed: {len(bundle_local)} traces, IDs: {bundle_local.ids}")

# To parse a real file:
# with open("data.mseed", "rb") as f:
#     bundle = parse_mseed(f.read())

## 8. Raw bytes for custom pipelines

If you want to handle miniSEED yourself (e.g., feed to a different decoder),
`get_raw()` gives you the bytes without any parsing.

In [ ]:
raw = client.get_raw(
    "IU",
    "ANMO",
    starttime="2024-01-15T00:00:00",
    endtime="2024-01-15T00:01:00",
)
print(f"Raw bytes: {len(raw):,}")
print(f"First 20 bytes: {raw[:20]}")

## Summary

| Method | Returns | Deps |
|--------|---------|------|
| `get_raw()` | `bytes` | core |
| `get_numpy()` | `TraceBundle` → `.to_dict()` = `{nslc: ndarray}` | core |
| `get_xarray()` | `xarray.Dataset` | + xarray |
| `get_waveforms()` | `obspy.Stream` | + obspy |
| `parse_mseed(bytes)` | `TraceBundle` | core |